# 决策树完整实现示例

## 目标

深入理解决策树的工作原理和应用。

- 理决策树的分裂标准和剪枝策略
- 从零实现决策树算法
- 与 Scikit-learn 的对比
- 可视化决策树结构

## 1. 数据准备

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, make_classification
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import pandas as pd

np.random.seed(42)

# 加载 Iris 数据集
iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

print(f"数据集形状: {X.shape}")
print(f"类别数量: {len(target_names)}")
print(f"类别名称: {target_names}")
print(f"特征名称: {feature_names}")

In [ ]:
# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"训练集: {X_train.shape}")
print(f"测试集: {X_test.shape}")

## 2. 决策树基础原理

In [ ]:
# 信息熵计算
def entropy(y):
    """
    计算信息熵
    
    H(S) = -Σ p_i * log2(p_i)
    """
    _, counts = np.unique(y, return_counts=True)
    probabilities = counts / len(y)
    
    # 过滤掉概率为0的情况
    probabilities = probabilities[probabilities > 0]
    
    return -np.sum(probabilities * np.log2(probabilities))

# 基尼不纯度计算
def gini_impurity(y):
    """
    计算基尼不纯度
    
    G(S) = 1 - Σ p_i^2
    """
    _, counts = np.unique(y, return_counts=True)
    probabilities = counts / len(y)
    return 1 - np.sum(probabilities ** 2)

# 信息增益计算
def information_gain(y, left_indices, right_indices, criterion='entropy'):
    """
    计算信息增益
    
    IG(S, A) = H(S) - Σ (|S_v| / |S|) * H(S_v)
    """
    if criterion == 'entropy':
        parent_impurity = entropy(y)
        left_impurity = entropy(y[left_indices])
        right_impurity = entropy(y[right_indices])
    else:
        parent_impurity = gini_impurity(y)
        left_impurity = gini_impurity(y[left_indices])
        right_impurity = gini_impurity(y[right_indices])
    
    n = len(y)
    n_left = len(left_indices)
    n_right = len(right_indices)
    
    weighted_impurity = (n_left / n) * left_impurity + (n_right / n) * right_impurity
    
    return parent_impurity - weighted_impurity

print("决策树分裂标准函数定义完成")

# 示例：计算整体数据集的熵和基尼不纯度
print(f"\n整体数据集的信息熵: {entropy(y):.4f}")
print(f"整体数据集的基尼不纯度: {gini_impurity(y):.4f}")

# 计算每个类别的分布
unique, counts = np.unique(y, return_counts=True)
print(f"\n类别分布:")
for i, (u, c) in enumerate(zip(unique, counts)):
    print(f"  {target_names[u]}: {c} ({c/len(y):.1%})")

## 3. 从零实现简单的决策树

In [ ]:
class SimpleDecisionTree:
    """
    简单的决策树实现（用于二分类，仅支持基尼系数）
    """
    
    def __init__(self, max_depth=None, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.tree = None
        self.feature_importances_ = None
    
    def fit(self, X, y):
        """训练决策树"""
        n_features = X.shape[1]
        self.feature_importances_ = np.zeros(n_features)
        self.tree = self._grow_tree(X, y, depth=0)
        
        # 归一化特征重要性
        if np.sum(self.feature_importances_) > 0:
            self.feature_importances_ /= np.sum(self.feature_importances_)
        
        return self
    
    def _grow_tree(self, X, y, depth):
        """递归构建决策树"""
        n_samples, n_features = X.shape
        n_classes = len(np.unique(y))
        
        # 终止条件
        if (self.max_depth is not None and depth >= self.max_depth or
            n_samples < self.min_samples_split or
            n_classes == 1):
            return {'class': self._most_common_label(y)}
        
        # 寻找最佳分裂
        best_feature, best_threshold, best_gain = self._find_best_split(X, y)
        
        # 如果没有增益，直接返回
        if best_gain == 0:
            return {'class': self._most_common_label(y)}
        
        # 更新特征重要性
        self.feature_importances_[best_feature] += best_gain * n_samples
        
        # 分裂数据
        left_indices = X[:, best_feature] <= best_threshold
        right_indices = ~left_indices
        
        # 递归构建子树
        left_subtree = self._grow_tree(X[left_indices], y[left_indices], depth + 1)
        right_subtree = self._grow_tree(X[right_indices], y[right_indices], depth + 1)
        
        return {
            'feature': best_feature,
            'threshold': best_threshold,
            'left': left_subtree,
            'right': right_subtree
        }
    
    def _find_best_split(self, X, y):
        """寻找最佳分裂特征和阈值"""
        best_feature = None
        best_threshold = None
        best_gain = -1
        
        parent_impurity = gini_impurity(y)
        n_samples, n_features = X.shape
        
        for feature in range(n_features):
            # 获取该特征的所有唯一值作为候选阈值
            thresholds = np.unique(X[:, feature])
            
            for threshold in thresholds:
                left_indices = X[:, feature] <= threshold
                right_indices = ~left_indices
                
                # 如果分裂后一边为空，跳过
                if np.sum(left_indices) == 0 or np.sum(right_indices) == 0:
                    continue
                
                # 计算增益
                left_impurity = gini_impurity(y[left_indices])
                right_impurity = gini_impurity(y[right_indices])
                
                n_left = np.sum(left_indices)
                n_right = np.sum(right_indices)
                
                weighted_impurity = (n_left / n_samples) * left_impurity + (n_right / n_samples) * right_impurity
                gain = parent_impurity - weighted_impurity
                
                if gain > best_gain:
                    best_gain = gain
                    best_feature = feature
                    best_threshold = threshold
        
        return best_feature, best_threshold, best_gain
    
    def _most_common_label(self, y):
        """返回最常见的标签"""
        values, counts = np.unique(y, return_counts=True)
        return values[np.argmax(counts)]
    
    def predict(self, X):
        """预测"""
        return np.array([self._predict_sample(x, self.tree) for x in X])
    
    def _predict_sample(self, x, node):
        """预测单个样本"""
        if 'class' in node:
            return node['class']
        
        if x[node['feature']] <= node['threshold']:
            return self._predict_sample(x, node['left'])
        else:
            return self._predict_sample(x, node['right'])

print("简单决策树类定义完成")

## 4. 训练自定义决策树

In [ ]:
# 训练自定义决策树
custom_tree = SimpleDecisionTree(max_depth=3)
custom_tree.fit(X_train, y_train)

# 预测
y_pred_custom = custom_tree.predict(X_test)

print("自定义决策树结果:")
print(f"训练集准确率: {accuracy_score(y_train, custom_tree.predict(X_train)):.4f}")
print(f"测试集准确率: {accuracy_score(y_test, y_pred_custom):.4f}")

print("\n特征重要性:")
for i, (name, importance) in enumerate(zip(feature_names, custom_tree.feature_importances_)):
    print(f"  {name}: {importance:.4f}")

## 5. 使用 Scikit-learn 的决策树

In [ ]:
# 使用 Scikit-learn 的决策树
sklearn_tree = DecisionTreeClassifier(max_depth=3, random_state=42)
sklearn_tree.fit(X_train, y_train)

# 预测
y_pred_sklearn = sklearn_tree.predict(X_test)

print("Scikit-learn 决策树结果:")
print(f"训练集准确率: {accuracy_score(y_train, sklearn_tree.predict(X_train)):.4f}")
print(f"测试集准确率: {accuracy_score(y_test, y_pred_sklearn):.4f}")

print("\n特征重要性:")
for name, importance in zip(feature_names, sklearn_tree.feature_importances_):
    print(f"  {name}: {importance:.4f}")

## 6. 可视化决策树结构

In [ ]:
plt.figure(figsize=(20, 10))
plot_tree(sklearn_tree, 
          feature_names=feature_names, 
          class_names=target_names,
          filled=True, 
          rounded=True,
          fontsize=10)
plt.title('决策树结构可视化 (max_depth=3)', fontsize=16)
plt.tight_layout()
plt.show()

## 7. 不同深度的影响

In [ ]:
# 测试不同深度的决策树
max_depths = [1, 2, 3, 4, 5, 6, None]  # None 表示无限制

train_scores = []
test_scores = []

for depth in max_depths:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_train, y_train)
    
    train_scores.append(tree.score(X_train, y_train))
    test_scores.append(tree.score(X_test, y_test))

# 可视化
plt.figure(figsize=(10, 6))
depth_labels = [str(d) if d is not None else 'None' for d in max_depths]
plt.plot(depth_labels, train_scores, 'b-o', label='训练集', linewidth=2, markersize=8)
plt.plot(depth_labels, test_scores, 'r-o', label='测试集', linewidth=2, markersize=8)
plt.xlabel('最大深度 (max_depth)')
plt.ylabel('准确率')
plt.title('决策树深度对性能的影响')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# 找到最佳深度
best_idx = np.argmax(test_scores)
print(f"最佳深度: {max_depths[best_idx] if max_depths[best_idx] is not None else '无限制'}")
print(f"最佳测试集准确率: {test_scores[best_idx]:.4f}")

## 8. 最小样本分裂数的影响

In [ ]:
# 测试不同 min_samples_split
min_samples_splits = [2, 5, 10, 20, 50]

train_scores_split = []
test_scores_split = []

for min_split in min_samples_splits:
    tree = DecisionTreeClassifier(min_samples_split=min_split, random_state=42)
    tree.fit(X_train, y_train)
    
    train_scores_split.append(tree.score(X_train, y_train))
    test_scores_split.append(tree.score(X_test, y_test))

# 可视化
plt.figure(figsize=(10, 6))
plt.plot(min_samples_splits, train_scores_split, 'b-o', label='训练集', linewidth=2, markersize=8)
plt.plot(min_samples_splits, test_scores_split, 'r-o', label='测试集', linewidth=2, markersize=8)
plt.xlabel('最小样本分裂数 (min_samples_split)')
plt.ylabel('准确率')
plt.title('最小样本分裂数对性能的影响')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xscale('log')
plt.show()

# 找到最佳 min_samples_split
best_idx = np.argmax(test_scores_split)
print(f"最佳 min_samples_split: {min_samples_splits[best_idx]}")
print(f"最佳测试集准确率: {test_scores_split[best_idx]:.4f}")

## 9. 不同分裂标准对比

In [ ]:
# 对比基尼系数和信息增益
criterions = ['gini', 'entropy']

for criterion in criterions:
    tree = DecisionTreeClassifier(criterion=criterion, max_depth=3, random_state=42)
    tree.fit(X_train, y_train)
    
    y_pred = tree.predict(X_test)
    train_score = tree.score(X_train, y_train)
    test_score = tree.score(X_test, y_test)
    
    print(f"\n{criterion.upper()} 标准:")
    print(f"  训练集准确率: {train_score:.4f}")
    print(f"  测试集准确率: {test_score:.4f}")
    
    # 分类报告
    print(f"\n  分类报告:")
    print(classification_report(y_test, y_pred, target_names=target_names, digits=3))

## 10. 混淆矩阵可视化

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

# 训练最佳模型
best_tree = DecisionTreeClassifier(max_depth=3, random_state=42)
best_tree.fit(X_train, y_train)
y_pred_best = best_tree.predict(X_test)

# 混淆矩阵
cm = confusion_matrix(y_test, y_pred_best)

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 数值形式
im = axes[0].imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
axes[0].set_title('混淆矩阵 (数值形式)')
tick_marks = np.arange(len(target_names))
axes[0].set_xticks(tick_marks)
axes[0].set_yticks(tick_marks)
axes[0].set_xticklabels(target_names)
axes[0].set_yticklabels(target_names)
axes[0].set_xlabel('预测标签')
axes[0].set_ylabel('真实标签')

# 添加数值
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        axes[0].text(j, i, format(cm[i, j], 'd'),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > cm.max() / 2 else "black")

plt.colorbar(im, ax=axes[0])

# 归一化形式
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
im2 = axes[1].imshow(cm_normalized, interpolation='nearest', cmap=plt.cm.Blues)
axes[1].set_title('混淆矩阵 (归一化)')
axes[1].set_xticks(tick_marks)
axes[1].set_yticks(tick_marks)
axes[1].set_xticklabels(target_names)
axes[1].set_yticklabels(target_names)
axes[1].set_xlabel('预测标签')
axes[1].set_ylabel('真实标签')

for i in range(cm_normalized.shape[0]):
    for j in range(cm_normalized.shape[1]):
        axes[1].text(j, i, format(cm_normalized[i, j], '.2f'),
                 horizontalalignment="center",
                 color="white" if cm_normalized[i, j] > cm_normalized.max() / 2 else "black")

plt.colorbar(im2, ax=axes[1])
plt.tight_layout()
plt.show()

## 11. 决策边界可视化（2D数据）

In [ ]:
# 生成2D数据用于可视化决策边界
X_2d, y_2d = make_classification(
    n_samples=200,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    random_state=42,
    n_clusters_per_class=1
)

# 训练决策树
tree_2d = DecisionTreeClassifier(max_depth=4, random_state=42)
tree_2d.fit(X_2d, y_2d)

# 绘制决策边界
def plot_decision_boundary(model, X, y):
    h = 0.02  # 步长
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    plt.figure(figsize=(10, 8))
    plt.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.RdYlBu)
    plt.scatter(X[y == 0][:, 0], X[y == 0][:, 1], c='blue', label='类别 0', alpha=0.6)
    plt.scatter(X[y == 1][:, 0], X[y == 1][:, 1], c='red', label='类别 1', alpha=0.6)
    plt.xlabel('特征 1')
    plt.ylabel('特征 2')
    plt.title(f'决策树决策边界 (max_depth={model.max_depth})')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

plot_decision_boundary(tree_2d, X_2d, y_2d)

print(f"2D数据集准确率: {tree_2d.score(X_2d, y_2d):.4f}")

## 12. 不同深度的决策边界

In [ ]:
# 对比不同深度的决策边界
depths = [1, 3, 5, None]

plt.figure(figsize=(16, 4))

for i, depth in enumerate(depths):
    plt.subplot(1, 4, i+1)
    
    tree_temp = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree_temp.fit(X_2d, y_2d)
    
    h = 0.02
    x_min, x_max = X_2d[:, 0].min() - 1, X_2d[:, 0].max() + 1
    y_min, y_max = X_2d[:, 1].min() - 1, X_2d[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    Z = tree_temp.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    plt.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.RdYlBu)
    plt.scatter(X_2d[y_2d == 0][:, 0], X_2d[y_2d == 0][:, 1], c='blue', alpha=0.5)
    plt.scatter(X_2d[y_2d == 1][:, 0], X_2d[y_2d == 1][:, 1], c='red', alpha=0.5)
    
    depth_str = str(depth) if depth is not None else 'None'
    acc = tree_temp.score(X_2d, y_2d)
    plt.title(f'depth={depth_str}\n准确率={acc:.3f}')
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 13. 剪枝（预剪枝 vs 后剪枝）

In [ ]:
# 预剪枝（通过参数控制）
print("预剪枝参数:")
print("- max_depth: 限制树的最大深度")
print("- min_samples_split: 分裂所需的最小样本数")
print("- min_samples_leaf: 叶节点所需的最小样本数")
print("- max_leaf_nodes: 最大叶节点数")
print("- min_impurity_decrease: 分裂所需的最小不纯度减少量")

# 后剪枝（使用 cost complexity pruning）
from sklearn.tree import DecisionTreeClassifier

# 先训练一个无限制的树
tree_full = DecisionTreeClassifier(random_state=42)
tree_full.fit(X_train, y_train)

# 获取剪枝路径
path = tree_full.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas
impurities = path.impurities

print(f"\n剪枝路径包含 {len(ccp_alphas)} 个不同的 alpha 值")
print(f"Alpha 范围: [{ccp_alphas[0]:.6f}, {ccp_alphas[-1]:.6f}]")

In [ ]:
# 可视化剪枝路径
plt.figure(figsize=(10, 6))
plt.plot(ccp_alphas, impurities, marker='o', drawstyle="steps-post")
plt.xlabel('有效 alpha')
plt.ylabel('总不纯度')
plt.title('不纯度随 alpha 的变化')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# 训练不同 alpha 下的决策树
clfs = []
train_scores_alpha = []
test_scores_alpha = []

for ccp_alpha in ccp_alphas:
    clf = DecisionTreeClassifier(random_state=42, ccp_alpha=ccp_alpha)
    clf.fit(X_train, y_train)
    clfs.append(clf)
    train_scores_alpha.append(clf.score(X_train, y_train))
    test_scores_alpha.append(clf.score(X_test, y_test))

# 去掉最后一个（只有一个节点）
clfs = clfs[:-1]
ccp_alphas = ccp_alphas[:-1]
train_scores_alpha = train_scores_alpha[:-1]
test_scores_alpha = test_scores_alpha[:-1]

# 可视化
plt.figure(figsize=(10, 6))
plt.plot(ccp_alphas, train_scores_alpha, marker='o', label='训练集', drawstyle="steps-post")
plt.plot(ccp_alphas, test_scores_alpha, marker='o', label='测试集', drawstyle="steps-post")
plt.xlabel('alpha')
plt.ylabel('准确率')
plt.title('准确率随 alpha 的变化（后剪枝）')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# 找到最佳 alpha
best_alpha_idx = np.argmax(test_scores_alpha)
best_alpha = ccp_alphas[best_alpha_idx]
print(f"最佳 alpha: {best_alpha:.6f}")
print(f"最佳测试集准确率: {test_scores_alpha[best_alpha_idx]:.4f}")
print(f"对应的树深度: {clfs[best_alpha_idx].tree_.max_depth}")
print(f"对应的叶节点数: {clfs[best_alpha_idx].tree_.n_leaves}")

## 14. 交叉验证选择最佳参数

In [ ]:
from sklearn.model_selection import GridSearchCV

# 定义参数网格
param_grid = {
    'max_depth': [2, 3, 4, 5, 6, None],
    'min_samples_split': [2, 5, 10, 20],
    'criterion': ['gini', 'entropy']
}

# 网格搜索
grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("最佳参数:")
print(f"  {grid_search.best_params_}")
print(f"\n最佳交叉验证得分: {grid_search.best_score_:.4f}")
print(f"测试集准确率: {grid_search.score(X_test, y_test):.4f}")

## 总结

### 决策树的关键特点

| 优点 | 缺点 |
|------|------|
| 易于理解和解释 | 容易过拟合 |
| 不需要特征缩放 | 不稳定（数据小变化可能产生完全不同的树）|
| 可以处理数值和类别特征 | 对数据不平衡敏感 |
| 自动特征选择 | 难以建模复杂关系（如 XOR）|

### 关键参数

1. **分裂标准**:
   - `criterion='gini'`: 基尼系数（默认）
   - `criterion='entropy'`: 信息增益

2. **树结构控制**:
   - `max_depth`: 最大深度
   - `min_samples_split`: 分裂所需最小样本数
   - `min_samples_leaf`: 叶节点最小样本数
   - `max_leaf_nodes`: 最大叶节点数

3. **剪枝**:
   - 预剪枝：通过上述参数控制
   - 后剪枝：`ccp_alpha` 参数

### 实践建议

- 默认使用 `max_depth` 控制过拟合
- 使用交叉验证选择最佳参数
- 可视化决策树以理解模型决策
- 考虑特征重要性进行特征选择
- 对于复杂任务，考虑集成方法（随机森林、梯度提升）